In [ ]:
import torch                                 #to import Pytorch i.e it handles tensors, gradients and CPU computations
import torch.nn as nn                        #nn stands for neural netwrok model i.e it contains layers (linear ,conv2d) ,Activations and lost functions(CrossEntropyLoss)
import torch.optim as optim                  #Optim stands for optimizers which is a algorithm that updates weights
from torchvision import datasets, transforms #To import ready made datasets like MNIST , transforms = to preprocess functions
from torch.utils.data import DataLoader      #To import batching ,shuffling and Parallel Loading i.e instead of feeding 60k images at once , we feed small datasets

#Model Definition : 
# We define a custom neural netwrok and inherits it from nn.Module
# This gives Parameter tracking, Backprop support
class MNISTNet(nn.Module):
    # init runs when model is created , self init initializes the parent class
    def __init__(self):
        super().__init__()

            # this is to create layers
            # MNISt images are 2D = 28*28 , for fully connected layers we must convert it to 1d 
            # so it converts [1,28,28] --> [784]
        self.flatten = nn.Flatten()

           # To get the number of neurons from a fully connected layer 
           #input = 28 , output = 128
           #each neurons learns a diff pattern
        self.fc1 = nn.Linear(28 * 28, 128)
           # This is a activation function , which adds non linearity , as w/o ReLU netwrok becomes useless
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

#
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root="data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

#
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MNISTNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

#
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}")

#
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predictions == labels).sum().item()

print(f"Test Accuracy: {correct / total:.4f}")

#
torch.save(model.state_dict(), "mnist_model.pth")
print("Model saved as mnist_model.pth")
